# Diagnóstico inicial de calidad de datos: Instacart

**Actividad:** GA1-220501094-01-AA1-EV04 · Fundamentos en Pandas  
**Aprendiz:** Carlos Eduardo Herrera Sibaja  
**Programa:** Análisis y Desarrollo de Software  
**Fecha:** 04/11/2026

## Propósito y pregunta analítica
Inspeccionar los cinco archivos seleccionados para reconocer su estructura, valores ausentes, tipos, posibles duplicados y consistencia básica de relaciones antes de continuar el análisis. La pregunta del proyecto es: **¿Qué patrones descriptivos se observan en los pedidos y artículos de Instacart, en particular en el horario de compra y la recompra de productos?**

Una fila de `order_products` representa un artículo dentro de un pedido, mientras que una fila de `instacart_orders` representa un pedido. Esta diferencia de granularidad debe considerarse al interpretar conteos.

## Aspectos de calidad a revisar
1. Dimensiones, nombres de columnas y tipos de datos de cada archivo.
2. Valores faltantes por columna y su significado.
3. Filas duplicadas y unicidad de los identificadores (`order_id`, `product_id`, `aisle_id`, `department_id`).
4. Rangos válidos: hora (0–23), día de la semana (0–6), días desde el pedido anterior (0–30) y `reordered` (0/1).
5. Integridad de las claves entre archivos y categorías de relleno en los catálogos.

In [1]:
import warnings
from pathlib import Path

import pandas as pd

DATA_DIR = Path('../data/raw')
if not DATA_DIR.exists():
    DATA_DIR = Path('data/raw')

archivos = {
    'orders': 'instacart_orders.csv',
    'order_products': 'order_products.csv',
    'products': 'products.csv',
    'aisles': 'aisles.csv',
    'departments': 'departments.csv',
}
ausentes = [nombre for nombre in archivos.values() if not (DATA_DIR / nombre).exists()]
if ausentes:
    raise FileNotFoundError(
        f'Faltan archivos en {DATA_DIR.as_posix()}: {ausentes}. '
        'Descargue los cinco CSV y colóquelos en data/raw/.'
    )

dataframes = {
    nombre: pd.read_csv(DATA_DIR / archivo, sep=';')
    for nombre, archivo in archivos.items()
}
print(f'Archivos cargados: {len(dataframes)}')
print('Directorio de datos:', DATA_DIR.as_posix())

Archivos cargados: 5
Directorio de datos: ../data/raw


## 1. Inspección estructural por dataset
Para cada DataFrame se aplican `head()`, `tail()`, `shape`, `columns`, `dtypes`, `info()`, `describe()` y `describe(include='object')` cuando existan columnas de texto. Al final se presenta la tabla de diagnóstico estructural (Dataset, Filas, Columnas, Tipos de datos, Observaciones).

In [2]:
resumen_estructural = []
for nombre, df in dataframes.items():
    print(f'\n{"=" * 80}\nDATASET: {nombre}\n{"=" * 80}')
    print('Primeras filas:')
    display(df.head())
    print('Últimas filas:')
    display(df.tail())
    print('Dimensiones (shape):', df.shape)
    print('Columnas:', df.columns.tolist())
    print('Tipos de datos:\n', df.dtypes)
    print('Información general:')
    df.info()
    print('Estadísticas descriptivas de variables numéricas:')
    display(df.describe())
    if not df.select_dtypes(include=['object', 'str']).columns.empty:
        print('Estadísticas descriptivas de variables de texto:')
        # En pandas 3 el texto se carga como 'str'; include='object' aún lo incluye, con un aviso de compatibilidad.
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', pd.errors.Pandas4Warning)
            display(df.describe(include='object'))

    nulos = df.isna().sum()
    observaciones = [f'{col}: {n:,} nulos' for col, n in nulos[nulos > 0].items()]
    duplicados = int(df.duplicated().sum())
    if duplicados:
        observaciones.append(f'{duplicados} filas duplicadas exactas')
    resumen_estructural.append({
        'Dataset': archivos[nombre],
        'Filas': df.shape[0],
        'Columnas': df.shape[1],
        'Tipos de datos': ', '.join(f'{col} ({tipo})' for col, tipo in df.dtypes.astype(str).items()),
        'Observaciones': '; '.join(observaciones) or 'Sin nulos ni duplicados exactos',
    })

tabla_estructural = pd.DataFrame(resumen_estructural)
with pd.option_context('display.max_colwidth', None):
    display(tabla_estructural)


DATASET: orders
Primeras filas:


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
1,1690866,163593,5,5,12,9.0
2,1454967,39980,4,5,19,2.0
3,1768857,82516,56,0,20,10.0
4,3007858,196724,2,4,12,17.0


Últimas filas:


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
478962,3210681,5617,5,1,14,7.0
478963,3270802,112087,2,3,13,6.0
478964,885349,82944,16,2,11,6.0
478965,216274,4391,3,3,8,8.0
478966,2071924,1730,18,1,14,15.0


Dimensiones (shape): (478967, 6)
Columnas: ['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']
Tipos de datos:
 order_id                    int64
user_id                     int64
order_number                int64
order_dow                   int64
order_hour_of_day           int64
days_since_prior_order    float64
dtype: object
Información general:
<class 'pandas.DataFrame'>
RangeIndex: 478967 entries, 0 to 478966
Data columns (total 6 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   order_id                478967 non-null  int64  
 1   user_id                 478967 non-null  int64  
 2   order_number            478967 non-null  int64  
 3   order_dow               478967 non-null  int64  
 4   order_hour_of_day       478967 non-null  int64  
 5   days_since_prior_order  450148 non-null  float64
dtypes: float64(1), int64(5)
memory usage: 21.9 MB
Estadísticas descri

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
count,4.789670e+05,478967.000000,478967.000000,478967.000000,478967.000000,450148.000000
mean,1.709947e+06,102966.537475,17.157808,2.775058,13.446676,11.101751
std,9.869480e+05,59525.328991,17.704688,2.045869,4.224986,9.189975
min,4.000000e+00,2.000000,1.000000,0.000000,0.000000,0.000000
25%,8.564685e+05,51414.000000,5.000000,1.000000,10.000000,4.000000
50%,1.709522e+06,102633.000000,11.000000,3.000000,13.000000,7.000000
75%,2.562990e+06,154419.000000,23.000000,5.000000,16.000000,15.000000
max,3.421079e+06,206209.000000,100.000000,6.000000,23.000000,30.000000


... (27 outputs left out)

,department_id
count,21.000000
mean,11.000000
std,6.204837
min,1.000000
25%,6.000000
50%,11.000000
75%,16.000000
max,21.000000


Estadísticas descriptivas de variables de texto:


,department
count,21
unique,21
top,frozen
freq,1


,Dataset,Filas,Columnas,Tipos de datos,Observaciones
0,instacart_orders.csv,478967,6,"order_id (int64), user_id (int64), order_number (int64), order_dow (int64), order_hour_of_day (int64), days_since_prior_order (float64)","days_since_prior_order: 28,819 nulos; 15 filas duplicadas exactas"
1,order_products.csv,4545007,4,"order_id (int64), product_id (int64), add_to_cart_order (float64), reordered (int64)",add_to_cart_order: 836 nulos
2,products.csv,49694,4,"product_id (int64), product_name (str), aisle_id (int64), department_id (int64)","product_name: 1,258 nulos"
3,aisles.csv,134,2,"aisle_id (int64), aisle (str)",Sin nulos ni duplicados exactos
4,departments.csv,21,2,"department_id (int64), department (str)",Sin nulos ni duplicados exactos


## 2. Valores faltantes y filas duplicadas
Los nulos se cuantifican por columna. Un duplicado exacto se cuenta sobre todas las columnas; ese conteo es una señal para investigar, no una instrucción automática para eliminar registros.

In [3]:
resumen_calidad = []
for nombre, df in dataframes.items():
    print(f'\n{archivos[nombre]}')
    nulos = df.isna().sum().sort_values(ascending=False)
    display(nulos[nulos > 0].rename('valores_faltantes').to_frame())
    duplicados = int(df.duplicated().sum())
    print('Filas duplicadas exactas:', duplicados)
    resumen_calidad.append({
        'Dataset': archivos[nombre],
        'Filas duplicadas exactas': duplicados,
        'Columnas con valores faltantes': ', '.join(nulos[nulos > 0].index.tolist()) or 'Ninguna',
        'Cantidad de valores faltantes': int(nulos.sum()),
    })

display(pd.DataFrame(resumen_calidad))


instacart_orders.csv


,valores_faltantes
days_since_prior_order,28819


Filas duplicadas exactas: 15

order_products.csv


,valores_faltantes
add_to_cart_order,836


Filas duplicadas exactas: 0

products.csv


,valores_faltantes
product_name,1258


... (2 outputs left out)

Filas duplicadas exactas: 0

departments.csv


,valores_faltantes


Filas duplicadas exactas: 0


,Dataset,Filas duplicadas exactas,Columnas con valores faltantes,Cantidad de valores faltantes
0,instacart_orders.csv,15,days_since_prior_order,28819
1,order_products.csv,0,add_to_cart_order,836
2,products.csv,0,product_name,1258
3,aisles.csv,0,Ninguna,0
4,departments.csv,0,Ninguna,0


## 3. Rangos, unicidad e integridad de claves
Se revisan campos clave y dominios que pueden validarse sin modificar datos. Los resultados describen posibles inconsistencias; las reglas del negocio deben confirmarse antes de corregirlas.

In [4]:
orders = dataframes['orders']
order_products = dataframes['order_products']
products = dataframes['products']
aisles = dataframes['aisles']
departments = dataframes['departments']

def resumen_clave(df, columna):
    return {
        'Columna': columna,
        'Nulos': int(df[columna].isna().sum()),
        'Valores únicos': int(df[columna].nunique(dropna=True)),
        'Duplicados de clave': int(df[columna].duplicated().sum()),
        'Mínimo': df[columna].min(),
        'Máximo': df[columna].max(),
    }

claves = [
    resumen_clave(orders, 'order_id'),
    resumen_clave(products, 'product_id'),
    resumen_clave(aisles, 'aisle_id'),
    resumen_clave(departments, 'department_id'),
]
display(pd.DataFrame(claves))

print('Valores fuera del rango usual de hora [0, 23]:', int((~orders['order_hour_of_day'].between(0, 23)).sum()))
print('Valores fuera del rango usual de día [0, 6]:', int((~orders['order_dow'].between(0, 6)).sum()))
print('Valores de reordered fuera de {0, 1}:', int((~order_products['reordered'].isin([0, 1])).sum()))

claves_huerfanas = {
    'order_id de artículos sin pedido': int((~order_products['order_id'].isin(orders['order_id'])).sum()),
    'product_id de artículos sin producto': int((~order_products['product_id'].isin(products['product_id'])).sum()),
    'aisle_id de productos sin pasillo': int((~products['aisle_id'].isin(aisles['aisle_id'])).sum()),
    'department_id de productos sin departamento': int((~products['department_id'].isin(departments['department_id'])).sum()),
}
display(pd.Series(claves_huerfanas, name='filas con clave sin correspondencia').to_frame())

,Columna,Nulos,Valores únicos,Duplicados de clave,Mínimo,Máximo
0,order_id,0,478952,15,4,3421079
1,product_id,0,49694,0,1,49694
2,aisle_id,0,134,0,1,134
3,department_id,0,21,0,1,21


Valores fuera del rango usual de hora [0, 23]: 0
Valores fuera del rango usual de día [0, 6]: 0
Valores de reordered fuera de {0, 1}: 0


,filas con clave sin correspondencia
order_id de artículos sin pedido,0
product_id de artículos sin producto,0
aisle_id de productos sin pasillo,0
department_id de productos sin departamento,0


## 4. Verificación de hallazgos específicos
Se comprueban con Pandas las causas probables de los problemas detectados, para sustentar la tabla de hallazgos del informe.

In [5]:
# Duplicados en orders: ¿son filas idénticas o pedidos distintos con el mismo order_id?
duplicados_orders = orders[orders.duplicated(keep=False)].sort_values('order_id')
print('Filas duplicadas exactas en orders:', int(orders.duplicated().sum()))
print('order_id repetidos:', int(orders['order_id'].duplicated().sum()))
display(duplicados_orders.head(6))
print('Distribución por día y hora de las filas duplicadas:')
display(duplicados_orders[['order_dow', 'order_hour_of_day']].value_counts().to_frame('filas'))

# Nulos en days_since_prior_order frente a primeros pedidos
nulos_intervalo = orders['days_since_prior_order'].isna()
primeros = orders['order_number'] == 1
print('\nNulos en days_since_prior_order:', int(nulos_intervalo.sum()))
print('Pedidos con order_number == 1:', int(primeros.sum()))
print('Nulos que NO son primer pedido:', int((nulos_intervalo & ~primeros).sum()))
print('Rango de days_since_prior_order:', orders['days_since_prior_order'].min(), '-', orders['days_since_prior_order'].max())

# Nulos en add_to_cart_order frente al tamaño del pedido
articulos_por_pedido = order_products.groupby('order_id').size()
pedidos_con_nulos = order_products.loc[order_products['add_to_cart_order'].isna(), 'order_id'].unique()
print('\nNulos en add_to_cart_order:', int(order_products['add_to_cart_order'].isna().sum()))
print('Pedidos afectados:', len(pedidos_con_nulos))
print('Valor máximo registrado de add_to_cart_order:', order_products['add_to_cart_order'].max())
print('Tamaño mínimo de los pedidos afectados:', int(articulos_por_pedido.loc[pedidos_con_nulos].min()))
print('Valores no enteros en add_to_cart_order:',
      int((order_products['add_to_cart_order'].dropna() % 1 != 0).sum()))

# Nombres de producto ausentes y categorías 'missing'
sin_nombre = products[products['product_name'].isna()]
print('\nProductos sin nombre:', len(sin_nombre))
display(sin_nombre[['aisle_id', 'department_id']].value_counts().to_frame('productos'))
display(aisles[aisles['aisle'] == 'missing'])
display(departments[departments['department'] == 'missing'])

# Nombres repetidos al ignorar mayúsculas y espacios
nombres_normalizados = products['product_name'].dropna().str.lower().str.strip()
print('Nombres de producto repetidos (ignorando mayúsculas/espacios):', int(nombres_normalizados.duplicated().sum()))
print('Nombres repetidos de forma exacta:', int(products['product_name'].dropna().duplicated().sum()))

Filas duplicadas exactas en orders: 15
order_id repetidos: 15


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
354993,391768,57671,19,3,2,10.0
371905,391768,57671,19,3,2,10.0
119251,408114,68324,4,3,2,18.0
321100,408114,68324,4,3,2,18.0
394347,467134,63189,21,3,2,2.0
250626,467134,63189,21,3,2,2.0


Distribución por día y hora de las filas duplicadas:


,,filas
order_dow,order_hour_of_day,
3,2,30



Nulos en days_since_prior_order: 28819
Pedidos con order_number == 1: 28819
Nulos que NO son primer pedido: 0
Rango de days_since_prior_order: 0.0 - 30.0



Nulos en add_to_cart_order: 836
Pedidos afectados: 70
Valor máximo registrado de add_to_cart_order: 64.0
Tamaño mínimo de los pedidos afectados: 65
Valores no enteros en add_to_cart_order: 0

Productos sin nombre: 1258


,,productos
aisle_id,department_id,
100,21,1258


,aisle_id,aisle
99,100,missing


,department_id,department
20,21,missing


Nombres de producto repetidos (ignorando mayúsculas/espacios): 104
Nombres repetidos de forma exacta: 0


## 5. Interpretación y acciones propuestas

- **`instacart_orders`:** hay 15 filas duplicadas exactas (mismo `order_id` y mismos valores), todas el miércoles a las 2 a. m., lo que sugiere un error de carga. Deben eliminarse antes de contar pedidos, porque inflan las frecuencias por hora y día.
- **`instacart_orders`:** los 28.819 nulos de `days_since_prior_order` coinciden exactamente con los pedidos `order_number == 1`: es la primera compra del usuario, no un error. Conviene conservarlos como nulos y excluirlos solo de los cálculos de intervalo. No se deben imputar con 0.
- **`order_products`:** los 836 nulos de `add_to_cart_order` aparecen solo en pedidos con más de 64 artículos, y el valor máximo registrado es 64. Parece un límite del sistema de registro. La columna es `float64` por los nulos; puede convertirse a entero con nulos (`Int64`).
- **`products`:** los 1.258 productos sin nombre pertenecen todos al pasillo 100 y al departamento 21, que en los catálogos se llaman `missing`. Son productos sin clasificar; pueden etiquetarse como `Unknown` sin perder sus identificadores.
- **`products`:** hay 104 nombres que se repiten al ignorar mayúsculas y espacios. Es posible que sean el mismo producto con distinto `product_id`; normalizar el texto (minúsculas, sin espacios extra) antes de agrupar por nombre.
- **Integridad:** no hay claves huérfanas entre tablas, y los rangos de hora (0–23), día (0–6), intervalo (0–30) y `reordered` (0/1) son válidos.
- **Volumen:** `order_products` supera los 4,5 millones de filas; conviene leer solo las columnas necesarias.

### Conclusión
Los datos están en condiciones de usarse para la pregunta analítica después de una limpieza ligera: eliminar los 15 duplicados, documentar el significado de los nulos y normalizar los nombres de producto. Ninguno de los problemas encontrados impide analizar el horario de compra ni la recompra. Este cuaderno solo diagnostica y propone acciones; los datos originales se conservan sin modificar.